# XYZ Bank Digital Customers — SQL Data Cleaning

> **Author:** Benard Mwinzi  
> **Dataset:** XYZ Bank Digital Customers (2,012 records)  
> **Tools:** MySQL · PyMySQL · SQLAlchemy · Pandas · Jupyter Notebook  
> **Domain:** Banking & Financial Services Analytics

---

## Project Overview

This part of the project demonstrates end-to-end data cleaning on a real-world banking dataset containing
digital customer records for XYZ Bank. Raw customer data loaded from a CSV file is audited,
cleaned, and validated using MySQL, executed and documented inside a Jupyter Notebook for
full reproducibility and portfolio presentation.

---

## Goals & Objectives

| # | Goal | Technique |
|---|------|-----------|
| 1 | Create the database and table with correct data types | `CREATE DATABASE`, `CREATE TABLE` |
| 2 | Load raw CSV data cleanly into MySQL | `LOAD DATA LOCAL INFILE` + `IF()` date handling |
| 3 | Verify the load was successful | `COUNT(*)`, `SELECT LIMIT` |
| 4 | Audit nulls and blanks across key columns | `SUM(IS NULL)`, `SUM(= '')` |
| 5 | Identify inconsistencies in categorical columns | `GROUP BY` distinct value checks |
| 6 | Remove leading/trailing whitespace from segment | `TRIM()` + `UPDATE` |
| 7 | Replace blank status and region values with NULL | `UPDATE ... SET ... WHERE = ''` |
| 8 | Validate all cleaning steps | Row counts, re-audit, distinct value checks |

---
## 1. Environment Setup

We use `SQLAlchemy` + `PyMySQL` as the database connection layer and `pandas` to display
query results as DataFrames. Two helper functions keep every cell clean:
- **`query(sql)`** — runs a SELECT and returns a styled DataFrame
- **`execute(sql)`** — runs DDL / DML and prints rows affected

A separate raw `pymysql` connection (`conn`) is kept exclusively for `LOAD DATA LOCAL INFILE`,
which requires a direct DBAPI connection rather than SQLAlchemy.

In [ ]:
import pymysql
import pandas as pd
from sqlalchemy import create_engine, text

# --- CONFIG ---
USER = "root"
PASSWORD = "Benara-123"
HOST = "localhost"
DB = "xyz_bank"

# --- SQLAlchemy engine ---
engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB}",
    connect_args={"local_infile": True},
    pool_pre_ping=True  # avoids stale connection issues
)

# --- Raw PyMySQL connection (for LOAD DATA LOCAL INFILE) ---
conn = pymysql.connect(
    host=HOST,
    user=USER,
    password=PASSWORD,
    database=DB,
    local_infile=True,
    cursorclass=pymysql.cursors.DictCursor  # optional: returns dicts instead of tuples
)

# --- Helper: SELECT queries → DataFrame ---
def query(sql):
    with engine.connect() as connection:
        return pd.read_sql(text(sql), connection)

# --- Helper: DDL / DML → rows affected ---
def execute(sql):
    with engine.connect() as connection:
        result = connection.execute(text(sql))
        connection.commit()
        print(f"OK — rows affected: {result.rowcount}")

print("✅ Connected successfully.")

✅ Connected successfully.


---
## 2. Database & Table Setup

We create the `xyz_bank` database and the `digital_customers` table with explicit data types.

**Design decisions:**
- `registration_date` is defined as `DATE` — it is always populated and has a consistent format
- `first_transaction_date` and `last_transaction_date` are initially loaded as `VARCHAR(30)` 
  because some rows have blank values, which MySQL cannot directly parse into a `DATE` column
- These will be converted to `DATE` after the data is loaded and cleaned

In [4]:
execute('CREATE DATABASE IF NOT EXISTS xyz_bank;')

OK — rows affected: 1


In [5]:
execute('DROP TABLE IF EXISTS digital_customers;')

OK — rows affected: 0


In [6]:
execute("""
CREATE TABLE digital_customers (
    customer_id            VARCHAR(10),
    registration_date      DATE,
    segment                VARCHAR(30),
    region                 VARCHAR(20),
    channel_registered     VARCHAR(20),
    age_band               VARCHAR(10),
    first_transaction_date VARCHAR(30),
    last_transaction_date  VARCHAR(30),
    total_transactions     INT,
    total_value_ksh        DECIMAL(15,2),
    monthly_active_months  INT,
    status                 VARCHAR(10),
    preferred_product      VARCHAR(20)
)
""")

OK — rows affected: 0


In [7]:
# Confirm schema
query('DESCRIBE digital_customers;')

,Field,Type,Null,Key,Default,Extra
0,customer_id,varchar(10),YES,,None,
1,registration_date,date,YES,,None,
2,segment,varchar(30),YES,,None,
3,region,varchar(20),YES,,None,
4,channel_registered,varchar(20),YES,,None,
5,age_band,varchar(10),YES,,None,
6,first_transaction_date,varchar(30),YES,,None,
7,last_transaction_date,varchar(30),YES,,None,
8,total_transactions,int,YES,,None,
9,total_value_ksh,"decimal(15,2)",YES,,None,


---
## 3. Data Loading

We use `LOAD DATA LOCAL INFILE` to load the CSV directly from the local filesystem.

**Key techniques used:**
- `SET GLOBAL local_infile = 1` — enables client-side file loading
- `SET SESSION sql_mode = ''` — relaxes strict mode to handle edge cases in the CSV
- Date columns are captured as variables (`@registration_date`, etc.) and converted
  using `IF(col = '', NULL, DATE(col))` — this prevents the `Incorrect datetime value: ''`
  warning that occurs when MySQL tries to parse an empty string as a date

In [8]:
# Enable local infile and relax strict mode
with conn.cursor() as cur:
    cur.execute("SET GLOBAL local_infile = 1;")
    cur.execute("SET SESSION sql_mode = '';")
    conn.commit()
print('Session configured.')

Session configured.


In [ ]:
# Load the CSV using the raw pymysql connection
load_sql = """
LOAD DATA LOCAL INFILE 'D:/Data Science/Digital Banking Project/Data/digital_customers_raw.csv'
INTO TABLE digital_customers
FIELDS TERMINATED BY ',' ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
IGNORE 1 LINES
(
    customer_id,
    @registration_date,
    segment,
    region,
    channel_registered,
    age_band,
    @first_transaction_date,
    @last_transaction_date,
    total_transactions,
    total_value_ksh,
    monthly_active_months,
    status,
    preferred_product
)
SET
    registration_date      = IF(@registration_date = '',      NULL, DATE(@registration_date)),
    first_transaction_date = IF(@first_transaction_date = '', NULL, DATE(@first_transaction_date)),
    last_transaction_date  = IF(@last_transaction_date = '',  NULL, DATE(@last_transaction_date))
"""

with conn.cursor() as cur:
    cur.execute(load_sql)
    conn.commit()
    print(f'OK — rows loaded: {cur.rowcount}')

OK — rows loaded: 2012


---
## 4. Load Verification

Before any cleaning, we verify the data loaded correctly by checking the row count
and previewing the first few records.

In [11]:
# Total row count — should be 2,012
query('SELECT COUNT(*) AS total_rows FROM digital_customers;')

,total_rows
0,2012


In [12]:
# Preview first 5 rows
query('SELECT * FROM digital_customers LIMIT 5;')

,customer_id,registration_date,segment,region,channel_registered,age_band,first_transaction_date,last_transaction_date,total_transactions,total_value_ksh,monthly_active_months,status,preferred_product
0,CUS01318,2024-01-02,Mass Affluent,Nairobi,Agent,18-25,NaN,NaN,0,0.00,0,Dormant,Mobile Transfer
1,CUS00527,2024-09-16,Mass Affluent,Mombasa,Agent,36-45,2024-09-23,2024-12-27,12,124627.37,1,Active,Bill Pay
2,CUS00394,2024-08-17,Mass Affluent,Eldoret,USSD,36-45,2024-08-22,2024-09-04,23,125188.53,2,Churned,Bill Pay
3,CUS01406,2024-11-18,SME,Nairobi,Mobile App,18-25,2024-12-04,2024-12-06,11,671595.93,1,,Digital Loan
4,CUS00434,2024-10-10,Mass Market,Nakuru,Mobile App,18-25,2024-10-24,2024-11-11,2,4512.18,1,Dormant,Bill Pay


---
## 5. Data Quality Audit

Before cleaning, we audit the dataset to understand the full scope of quality issues.
This step drives all downstream cleaning decisions.

### 5a. Null & Blank Audit

In [13]:
# Null and blank audit across key columns
query("""
SELECT
    SUM(registration_date IS NULL)      AS null_registration_date,
    SUM(first_transaction_date IS NULL) AS null_first_transaction,
    SUM(last_transaction_date IS NULL)  AS null_last_transaction,
    SUM(total_value_ksh IS NULL)        AS null_total_value,
    SUM(status = '' OR status IS NULL)  AS blank_status,
    SUM(region = '' OR region IS NULL)  AS blank_region
FROM digital_customers
""")

,null_registration_date,null_first_transaction,null_last_transaction,null_total_value,blank_status,blank_region
0,0.0,506.0,506.0,0.0,10.0,15.0


### 5b. Duplicate Row Check

In [14]:
query("""
SELECT customer_id, registration_date, segment, 
       channel_registered, total_transactions,
       COUNT(*) AS occurrences
FROM digital_customers
GROUP BY customer_id, registration_date, segment, 
         channel_registered, total_transactions
HAVING COUNT(*) > 1;
      """)

,customer_id,registration_date,segment,channel_registered,total_transactions,occurrences
0,CUS01188,2024-04-04,Mass Market,USSD,11,2
1,CUS01814,2024-07-23,Mass Market,Mobile App,2,2
2,CUS01735,2024-04-28,SME,USSD,28,2
3,CUS00953,2024-04-12,Corporate,Branch,72,2
4,CUS00115,2024-01-12,SME,Mobile App,64,2
5,CUS01030,2024-11-16,Corporate,USSD,0,2
6,CUS00792,2024-06-24,SME,Mobile App,40,2
7,CUS01060,2024-08-26,Mass Affluent,USSD,0,2
8,CUS00341,2024-09-29,Corporate,Branch,0,2
9,CUS00645,2024-09-26,SME,USSD,9,2


12 customer records appear exactly twice in the dataset, accounting for 12 duplicate rows.
Every duplicate has `occurrences = 2`, consistent with a double-load during data staging.
These will be removed in the cleaning phase, retaining one record per customer.

### 5c. Categorical Column Distinct Value Checks

We inspect each categorical column for inconsistencies such as:
- Inconsistent casing (e.g. `Active` vs `active`)
- Leading or trailing spaces creating duplicate categories
- Blank values appearing as a separate category
- Unexpected or invalid values

In [15]:
# Segment distribution
query("""
SELECT segment, COUNT(*) AS count
FROM digital_customers
GROUP BY segment
ORDER BY count DESC
""")

,segment,count
0,Mass Market,1131
1,Mass Affluent,467
2,SME,307
3,Corporate,100
4,Mass Affluent,7


In [16]:
# Status distribution
query("""
SELECT status, COUNT(*) AS count
FROM digital_customers
GROUP BY status
ORDER BY count DESC
""")

,status,count
0,Dormant,921
1,Churned,675
2,Active,406
3,,10


In [18]:
# Region distribution
query("""
SELECT region, COUNT(*) AS count
FROM digital_customers
GROUP BY region
ORDER BY count DESC
""")

,region,count
0,Nairobi,896
1,Mombasa,405
2,Kisumu,310
3,Nakuru,231
4,Eldoret,155
5,,15


In [19]:
# Channel registered distribution
query("""
SELECT channel_registered, COUNT(*) AS count
FROM digital_customers
GROUP BY channel_registered
ORDER BY count DESC
""")

,channel_registered,count
0,Mobile App,925
1,USSD,470
2,Agent,413
3,Branch,204


In [20]:
# Age band distribution
query("""
SELECT age_band, COUNT(*) AS count
FROM digital_customers
GROUP BY age_band
ORDER BY count DESC
""")

,age_band,count
0,26-35,696
1,36-45,558
2,18-25,353
3,46-55,238
4,55+,147
5,,20


In [21]:
# Preferred product distribution
query("""
SELECT preferred_product, COUNT(*) AS count
FROM digital_customers
GROUP BY preferred_product
ORDER BY count DESC
""")

,preferred_product,count
0,Mobile Transfer,640
1,Bill Pay,481
2,Airtime,377
3,Merchant Pay,294
4,Digital Loan,220


---
### 5d. Logical Date Consistency Check

In [22]:
query("""
SELECT COUNT(*) AS impossible_dates
      FROM digital_customers
      WHERE first_transaction_date < registration_date;
""")

,impossible_dates
0,25


### Logical Date Consistency Check — Finding

25 records have a first_transaction_date that falls before the customer's 
registration_date. This is a logically impossible since a customer cannot transact 
before they are registered on the system. This is consistent with a system clock 
error or a batch load timing issue on the source system.

Decision: Set first_transaction_date to NULL for these 25 records. The customer 
record is preserved but the invalid date is removed rather than guessed.

---
### 5e. Negative Value Check

In [23]:
query("""
SELECT COUNT(*) AS negative_transactions
      FROM digital_customers
      WHERE total_transactions < 0;

""")

,negative_transactions
0,8


8 records have `negative total_transactions values`. Transaction counts cannot 
be negative — this is consistent with a data entry sign error where a negative 
symbol was incorrectly applied during manual entry or system export. 

Decision: Apply ABS() to correct the sign while preserving the magnitude. 
No data is lost — only the incorrect sign is fixed.

---
### 5f. Transaction Value Integrity Check

In [24]:
query("""
SELECT COUNT(*) AS value_mismatches
      FROM digital_customers
      WHERE total_transactions > 0 AND total_value_ksh = 0;

""")

,value_mismatches
0,40


*Findings:* 40 records show total_transactions > 0 but total_value_ksh = 0. 
This is a data integrity issue — transactions exist but monetary value 
was not captured, consistent with an ETL sync failure between the 
transaction count system and the value aggregation process.

Decision: These records are flagged for finance team review and will be 
excluded from any revenue analysis to avoid understating transaction values. 
The customer records themselves are retained for customer journey analysis.

---
### 5g. NULL Age Band Check

In [25]:
query("""
SELECT COUNT(*) AS null_age_band
      FROM digital_customers
      WHERE age_band IS NULL;

""")

,null_age_band
0,0


In [26]:
query("""
SELECT COUNT(*) AS null_age_band
      FROM digital_customers
      WHERE age_band = ""OR age_band IS NULL;

""")

,null_age_band
0,20


*Finding:* 20 records have a blank age_band value (empty string ''), which were loaded 
silently by MySQL instead of NULL. These were missed in the initial IS NULL 
audit, highlighting the importance of checking both NULL and empty string 
conditions simultaneously.

Decision: Label as 'Unknown' rather than NULL so these records appear 
explicitly in age band distributions during analysis and visualization, 
ensuring transparency in reporting.

---
## 6. Audit Summary — Issues Identified

The following data quality issues were identified and will be addressed in Phase 2 (Data Cleaning):

| # | Column | Issue | Fix |
|---|--------|-------|-----|
| 1 | `segment` | `' Mass Affluent'` has a leading space, creating a duplicate category alongside `'Mass Affluent'` | `TRIM()` + `UPDATE` |
| 2 | `status` | Blank values `''` appearing as a separate empty category | Replace `''` with `NULL` |
| 3 | `region` | Blank values `''` appearing as a separate empty category | Replace `''` with `NULL` |
| 4 | `first_transaction_date` | Stored as `VARCHAR` — needs converting to `DATE` after cleaning | `ALTER TABLE MODIFY COLUMN` |
| 5 | `last_transaction_date` | Stored as `VARCHAR` — needs converting to `DATE` after cleaning | `ALTER TABLE MODIFY COLUMN` |

> These issues will be systematically resolved in **Phase 2: Data Cleaning**.

---
## 7. Phase 2: Data Cleaning

Each cleaning step follows the pattern: **preview → apply → verify**.
This ensures every change is intentional and validated before moving on.

### 7.1 Fix Segment — Remove Leading Space from `' Mass Affluent'`

**Problem:** The value `' Mass Affluent'` has a leading space, creating a duplicate
category alongside `'Mass Affluent'`. This would cause incorrect counts in any
segment-level aggregation.

**Fix:** `TRIM()` removes all leading and trailing whitespace.

In [27]:
# Preview: confirm the offending value exists
query("""
SELECT segment, COUNT(*) AS count
FROM digital_customers
WHERE segment = ' Mass Affluent'
GROUP BY segment
""")

,segment,count
0,Mass Affluent,7


In [28]:
# Apply: trim the leading space
execute("""
UPDATE digital_customers
SET segment = TRIM(segment)
WHERE segment = ' Mass Affluent'
""")

OK — rows affected: 7


In [29]:
# Verify: ' Mass Affluent' should be gone, 'Mass Affluent' count should increase
query("""
SELECT segment, COUNT(*) AS count
FROM digital_customers
GROUP BY segment
ORDER BY count DESC
""")

,segment,count
0,Mass Market,1131
1,Mass Affluent,474
2,SME,307
3,Corporate,100


### 7.2 Fix Status — Replace Blank Values with NULL

**Problem:** Some rows have `status = ''` (empty string), which appears as a
separate unnamed category in GROUP BY results.

**Fix:** Replace empty strings with `NULL` — the correct SQL representation
for missing/unknown values. This makes missing data explicit and easier to
filter with `WHERE status IS NULL`.

In [30]:
# Preview: how many blank status rows exist?
query("""
SELECT COUNT(*) AS blank_status_count
FROM digital_customers
WHERE status = ''
""")

,blank_status_count
0,10


In [31]:
# Apply: replace blank with NULL
execute("""
UPDATE digital_customers
SET status = NULL
WHERE status = ''
""")

OK — rows affected: 10


In [32]:
# Verify: no blank status values should remain
query("""
SELECT status, COUNT(*) AS count
FROM digital_customers
GROUP BY status
ORDER BY count DESC
""")

,status,count
0,Dormant,921
1,Churned,675
2,Active,406
3,NaN,10


### 7.3 Fix Region — Replace Blank Values with NULL

**Problem:** Same issue as `status` — blank strings `''` create a spurious
empty category in region-level analysis.

**Fix:** Replace empty strings with `NULL`.

In [33]:
# Preview: how many blank region rows exist?
query("""
SELECT COUNT(*) AS blank_region_count
FROM digital_customers
WHERE region = ''
""")

,blank_region_count
0,15


In [34]:
# Apply: replace blank with NULL
execute("""
UPDATE digital_customers
SET region = NULL
WHERE region = ''
""")

OK — rows affected: 15


In [35]:
# Verify: no blank region values should remain
query("""
SELECT region, COUNT(*) AS count
FROM digital_customers
GROUP BY region
ORDER BY count DESC
""")

,region,count
0,Nairobi,896
1,Mombasa,405
2,Kisumu,310
3,Nakuru,231
4,Eldoret,155
5,NaN,15


### 7.4 Convert Date Columns from VARCHAR to DATE

**Problem:** `first_transaction_date` and `last_transaction_date` were loaded as
`VARCHAR(30)` to handle blank values during import. Now that blanks have been
resolved to `NULL`, we can safely convert them to proper `DATE` columns.

**Fix:** `ALTER TABLE MODIFY COLUMN` changes the data type. MySQL will
automatically cast the `YYYY-MM-DD` strings to `DATE`.

In [36]:
# Preview: confirm date format looks correct before converting
query("""
SELECT first_transaction_date, last_transaction_date
FROM digital_customers
WHERE first_transaction_date IS NOT NULL
LIMIT 5
""")

,first_transaction_date,last_transaction_date
0,2024-09-23,2024-12-27
1,2024-08-22,2024-09-04
2,2024-12-04,2024-12-06
3,2024-10-24,2024-11-11
4,2024-10-01,2024-11-22


In [37]:
# Apply: convert both columns to DATE type
execute("""
ALTER TABLE digital_customers
MODIFY COLUMN first_transaction_date DATE,
MODIFY COLUMN last_transaction_date  DATE
""")

OK — rows affected: 2012


In [38]:
# Verify: DESCRIBE should now show DATE for both columns
query('DESCRIBE digital_customers;')

,Field,Type,Null,Key,Default,Extra
0,customer_id,varchar(10),YES,,None,
1,registration_date,date,YES,,None,
2,segment,varchar(30),YES,,None,
3,region,varchar(20),YES,,None,
4,channel_registered,varchar(20),YES,,None,
5,age_band,varchar(10),YES,,None,
6,first_transaction_date,date,YES,,None,
7,last_transaction_date,date,YES,,None,
8,total_transactions,int,YES,,None,
9,total_value_ksh,"decimal(15,2)",YES,,None,


### 7.5 Remove Duplicate Rows

In [39]:
# Apply: Adding a row ID so MySQL can identify individual rows
execute("""
ALTER TABLE digital_customers
ADD COLUMN row_id INT AUTO_INCREMENT PRIMARY KEY
      """)
# Deleting duplicates, keeping the lowest row_id for each customer
execute("""
DELETE FROM digital_customers
        WHERE row_id NOT IN (
            SELECT MIN(row_id)
            FROM (SELECT * FROM digital_customers) AS dc_copy
        GROUP BY customer_id, registration_date, segment, 
                 channel_registered, total_transactions
        );
""")
# Dropping the helper column and verify row count
execute("""
ALTER TABLE digital_customers
        DROP COLUMN row_id;
""")

OK — rows affected: 0
OK — rows affected: 12
OK — rows affected: 2000


In [40]:
# Verify: Checking that the data now has 2000 rows instead of 2012
query("""
SELECT COUNT(*) AS total_rows
      FROM digital_customers;
""")

,total_rows
0,2000


**Result:** 12 duplicate rows removed. Dataset reduced from 2,012 to 2,000 rows. 
Data integrity confirmed — 2,000 unique customer records remain.

### 7.6 Fix Impossible Transaction Dates

In [41]:
# Preview: which customers are affected
query("""
SELECT customer_id, registration_date, 
      DATEDIFF(registration_date, first_transaction_date) as days_before_registration
      FROM digital_customers
      WHERE first_transaction_date < registration_date
      ORDER BY days_before_registration DESC
      LIMIT 10;
""")

,customer_id,registration_date,days_before_registration
0,CUS00063,2024-08-06,10
1,CUS00633,2024-01-10,10
2,CUS01697,2024-02-18,9
3,CUS00915,2024-05-22,9
4,CUS00839,2024-07-11,9
5,CUS01771,2024-11-22,9
6,CUS01596,2024-03-13,7
7,CUS01463,2024-04-14,7
8,CUS01039,2024-04-09,6
9,CUS01212,2024-06-06,6


The maximum date discrepancy is `10 days` before registration, suggesting a 
systematic system clock or batch processing error rather than random data entry 
mistakes. No records show extreme outliers.

In [42]:
# Apply: Setting those first transaction dates to NULL
execute("""
UPDATE digital_customers
        SET first_transaction_date = NULL
        WHERE first_transaction_date < registration_date;
""")

OK — rows affected: 25


In [43]:
# Verify that the impossible dates have been resolved
query("""
SELECT COUNT(*) AS impossible_dates
      FROM digital_customers
      WHERE first_transaction_date < registration_date;

""")

,impossible_dates
0,0


**Result:** 25 impossible dates corrected. first_transaction_date set to NULL 
for all records where the transaction preceded registration. 
0 impossible dates remain.

### 7.7 Fix Negative Transaction Counts

In [44]:
# Preview the affected records
query("""
      SELECT customer_id, total_transactions, total_value_ksh
      FROM digital_customers
      WHERE total_transactions < 0
      ORDER BY total_transactions;
      """)

,customer_id,total_transactions,total_value_ksh
0,CUS01235,-56,2316946.73
1,CUS01872,-30,1337643.31
2,CUS00090,-23,302690.20
3,CUS01453,-17,17548.56
4,CUS00469,-17,39069.74
5,CUS01096,-11,12060.54
6,CUS01646,-7,5118.03
7,CUS00632,-7,63316.76


Note: total_value_ksh is positive for all 8 affected records, confirming 
the issue is isolated to a sign error on the total_transactions field only. 
The underlying customer data is otherwise intact.

In [45]:
# Apply the negative transactions fix
execute("""
UPDATE digital_customers
        SET total_transactions = ABS(total_transactions)
        WHERE total_transactions < 0
""")

OK — rows affected: 8


In [46]:
# Verify that no negative transactions remain
query("""
SELECT COUNT(*) AS negative_transactions
      FROM digital_customers
      WHERE total_transactions < 0;
""")

,negative_transactions
0,0


**Result:** 8 negative transaction counts corrected using ABS(). 
0 negative values remain.

### 7.8 Flag Value/Transaction Mismatches

In [47]:
query("""
SELECT customer_id, segment, channel_registered,
      total_transactions, total_value_ksh
      FROM digital_customers
      WHERE total_transactions > 0 AND total_value_ksh = 0
      ORDER BY total_transactions DESC
      LIMIT 10;
""")

,customer_id,segment,channel_registered,total_transactions,total_value_ksh
0,CUS01599,Corporate,Mobile App,216,0.0
1,CUS00401,SME,Mobile App,79,0.0
2,CUS01618,Mass Affluent,Branch,66,0.0
3,CUS00107,SME,USSD,63,0.0
4,CUS00458,Mass Affluent,USSD,50,0.0
5,CUS00230,SME,USSD,41,0.0
6,CUS00137,SME,Mobile App,36,0.0
7,CUS00450,Mass Affluent,Mobile App,31,0.0
8,CUS01185,Mass Affluent,Mobile App,26,0.0
9,CUS01906,Mass Affluent,USSD,23,0.0


The 40 affected records are spread across multiple segments (Corporate, SME, 
Mass Affluent) and channels (Mobile App, USSD, Branch), confirming this is a 
system-wide ETL sync failure rather than a segment or channel-specific issue.
Notably, CUS01599 (Corporate) has 216 transactions with zero value. This 
represents a significant revenue gap that finance should prioritize for recovery.

In [48]:
# Adding a flag column to mark affected records
execute("""
   ALTER TABLE digital_customers
        ADD COLUMN value_flag VARCHAR(20) DEFAULT NULL;
     """)

OK — rows affected: 0


In [49]:
# Flagging the records with zero value but positive transactions
execute("""
UPDATE digital_customers
        SET value_flag = 'Review Required'
        WHERE total_transactions > 0 AND total_value_ksh = 0;
""")

OK — rows affected: 40


In [50]:
# Verifying that the flag has been applied correctly
query("""
SELECT value_flag, COUNT(*) AS count
      FROM digital_customers
      GROUP BY value_flag;
""")

,value_flag,count
0,NaN,1960
1,Review Required,40


**Result:** 40 records flagged as `Review Required`. These records will be excluded 
from revenue analysis but retained for customer journey analysis. 
Finance team should investigate and recover the missing transaction values 
from source systems.

### 7.9 Fix Blank Age Band Values

In [51]:
# Preview: how many null or blank age_band rows exist?
query("""
SELECT COUNT(*) AS null_age_band
      FROM digital_customers
      WHERE age_band IS NULL OR age_band = "";
""")

,null_age_band
0,20


In [52]:
# Apply: replace blanks with "Unknown"
execute("""
UPDATE digital_customers
        SET age_band = "Unknown"
        WHERE age_band IS NULL OR age_band = "";
""")

OK — rows affected: 20


In [53]:
# Verify: no null or blank age_band values remain
query("""
SELECT age_band, COUNT(*) AS count
FROM digital_customers
GROUP BY age_band
ORDER BY count DESC;
    """)

,age_band,count
0,26-35,691
1,36-45,556
2,18-25,351
3,46-55,236
4,55+,146
5,Unknown,20


**Result:** 20 blank age_band values relabelled as 'Unknown'. These records 
will appear explicitly in all age band distributions rather than being 
silently excluded from analysis.

---
## 8. Final Data Quality Validation

A comprehensive quality check on the fully cleaned dataset.

In [54]:
# Final row count — should be 2,000 (12 duplicates removed)
query('SELECT COUNT(*) AS total_rows FROM digital_customers;')

,total_rows
0,2000


In [55]:
# Final null and blanks audit
query("""
SELECT
    SUM(customer_id IS NULL)                        AS null_customer_id,
    SUM(registration_date IS NULL)                  AS null_registration_date,
    SUM(segment = '' OR segment IS NULL)            AS null_segment,
    SUM(region IS NULL)                             AS null_region,
    SUM(channel_registered IS NULL)                 AS null_channel,
    SUM(age_band = '' OR age_band IS NULL)          AS null_age_band,
    SUM(first_transaction_date IS NULL)             AS null_first_txn,
    SUM(last_transaction_date IS NULL)              AS null_last_txn,
    SUM(total_transactions < 0)                     AS negative_transactions,
    SUM(total_transactions > 0 
        AND total_value_ksh = 0)                    AS value_mismatch,
    SUM(status IS NULL)                             AS null_status
FROM digital_customers;
""")

,null_customer_id,null_registration_date,null_segment,null_region,null_channel,null_age_band,null_first_txn,null_last_txn,negative_transactions,value_mismatch,null_status
0,0.0,0.0,0.0,15.0,0.0,0.0,528.0,503.0,0.0,40.0,10.0


In [56]:
# Final categorical checks
query("""
SELECT 'segment' AS column_name, segment AS value, COUNT(*) AS count
      FROM digital_customers GROUP BY segment
UNION ALL
      SELECT 'status', status, COUNT(*) FROM digital_customers GROUP BY status
UNION ALL
      SELECT 'age_band', age_band, COUNT(*) FROM digital_customers GROUP BY age_band
UNION ALL
      SELECT 'region', region, COUNT(*) FROM digital_customers GROUP BY region
ORDER BY column_name, count DESC;
""")

,column_name,value,count
0,age_band,26-35,691
1,age_band,36-45,556
2,age_band,18-25,351
3,age_band,46-55,236
4,age_band,55+,146
5,age_band,Unknown,20
6,region,Nairobi,890
7,region,Mombasa,404
8,region,Kisumu,308
9,region,Nakuru,230


In [57]:
# Preview cleaned dataset
query('SELECT * FROM digital_customers LIMIT 10;')

,customer_id,registration_date,segment,region,channel_registered,age_band,first_transaction_date,last_transaction_date,total_transactions,total_value_ksh,monthly_active_months,status,preferred_product,value_flag
0,CUS00112,2024-05-25,Mass Market,Mombasa,Agent,26-35,None,None,0,0.00,0,Dormant,Digital Loan,None
1,CUS00353,2024-08-30,Mass Market,Mombasa,Branch,55+,2024-09-18,2024-11-22,4,9569.17,1,Dormant,Bill Pay,None
2,CUS01318,2024-01-02,Mass Affluent,Nairobi,Agent,18-25,None,None,0,0.00,0,Dormant,Mobile Transfer,None
3,CUS01135,2024-05-05,Mass Affluent,Nairobi,Mobile App,36-45,2024-05-22,2024-07-16,48,221643.54,4,Churned,Airtime,None
4,CUS00858,2024-08-16,Mass Affluent,Mombasa,Agent,18-25,None,None,0,0.00,0,Dormant,Digital Loan,None
5,CUS00675,2024-03-09,Mass Market,Kisumu,Branch,26-35,None,None,0,0.00,0,Dormant,Merchant Pay,None
6,CUS00527,2024-09-16,Mass Affluent,Mombasa,Agent,36-45,2024-09-23,2024-12-27,12,124627.37,1,Active,Bill Pay,None
7,CUS01458,2024-07-30,Corporate,Nairobi,Agent,26-35,2024-08-13,2024-08-28,85,24402644.50,3,Churned,Digital Loan,None
8,CUS00571,2024-01-01,Mass Market,Nairobi,Agent,46-55,2024-01-09,2024-10-13,12,15785.96,3,Dormant,Mobile Transfer,None
9,CUS00923,2024-05-06,Mass Market,Mombasa,Mobile App,36-45,2024-05-14,2024-07-23,10,7981.39,2,Churned,Bill Pay,None


---
## 9. Conclusion & Key Takeaways

This project cleaned a 2,012-row banking dataset, resolving all identified
data quality issues while preserving full data integrity.

### Cleaning Results Summary

| Issue | Before | After |
|-------|--------|-------|
| Duplicate segment category (`' Mass Affluent'`) | 2 categories | 1 category |
| Blank status values | Empty string category | `NULL` (missing) |
| Blank region values | Empty string category | `NULL` (missing) |
| `first_transaction_date` data type | `VARCHAR(30)` | `DATE` |
| `last_transaction_date` data type | `VARCHAR(30)` | `DATE` |
| Duplicate rows | 2,012 rows | 2,000 rows (12 removed) |
| Impossible transaction dates | 25 records | NULL (corrected) |
| Negative transaction counts | 8 records | Fixed with ABS() |
| Value/transaction mismatches | 40 records | Flagged 'Review Required' |
| Blank age band values | 20 records | Labelled 'Unknown' |

### SQL Skills Demonstrated

| Skill | Applied In |
|-------|------------|
| DDL: `CREATE DATABASE`, `CREATE TABLE`, `ALTER TABLE` | Schema design & type conversion |
| `LOAD DATA LOCAL INFILE` | Bulk CSV loading |
| `IF()`, `NULLIF()` | Conditional date handling during load |
| `UPDATE`, `SET`, `WHERE` | Data correction |
| `TRIM()` | Whitespace removal |
| `GROUP BY`, `COUNT()` | Categorical auditing |
| `SUM(IS NULL)` | Null auditing |
| `ABS()` | Sign correction |
| Data integrity validation | Row count checks at each step |

In [58]:
# Close connections
conn.close()
engine.dispose()
print('Connections closed.')

Connections closed.
